In [1]:
%autosave 60
%pip install --quiet -r requirements.txt

Autosaving every 60 seconds
Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
import os
import numpy as np
import random
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import defaultdict
from utils.hunt_data_loader import HuntDataLoader
from utils.train_eval import fit_3D, fit_3D_gan
from utils.loss_functions import recon_loss, ssim_L1_2d_loss
from models.alzheiminator_3d import ResidualUNet3D, Discriminator3D, Generator3D
from models.no_more_alzheimer_2d import UNet2D
from tqdm import tqdm

data_loader = HuntDataLoader()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load models

We load our best performing 2D model

In [3]:
best_2d = UNet2D(latent_dim=64, in_channels=1, out_channels=1, base_ch=32).to(device)
best_2d.load_state_dict(torch.load("out/unet_models/2d_unet_model_best.pt"))

<All keys matched successfully>

We load our best performing 3D model

In [4]:
best_3d = ResidualUNet3D(in_ch=1, base=32).to(device)
best_3d.load_state_dict(torch.load("out/unet_models/3d_unet_model_best.pt"))

<All keys matched successfully>

## Display evenly spaced slices for both models

In [ ]:
# TODO

## Calculate Loss for each model

In [6]:
# Same split as during training
_, _, test_pairs = data_loader.split_dataset_paths(seed=69)
print("We have", len(test_pairs), "test pairs")

We have 71 test pairs


Helper function taking a 3D input sensor and running a 2D model over all slices, and then combining them into an output volume

In [7]:
def get_volume_from_2d_pred(model, x_vol):
    model.eval()

    # --- Normalize input shape to (D, H, W) ---
    if x_vol.dim() == 5:        # (B=1, C=1, D, H, W)
        x_vol = x_vol.squeeze(0).squeeze(0)   # -> (D, H, W)
    elif x_vol.dim() == 4:      # (B=1, D, H, W)
        x_vol = x_vol.squeeze(0)             # -> (D, H, W)
    elif x_vol.dim() != 3:      # must be (D, H, W)
        raise ValueError(f"Expected x_vol with 3, 4, or 5 dims, got {x_vol.shape}")

    D = x_vol.shape[0]
    output_vol = None  # will be allocated from first output slice

    with torch.no_grad():
        for i in range(D):
            # (H, W) -> (1,1,H,W)
            x_slice = x_vol[i].unsqueeze(0).unsqueeze(0).to(device)

            # Forward pass; adapt unpacking if your model returns differently
            recon, mu, logvar = model(x_slice)  # recon: (1,1,H_out,W_out)
            recon_slice = recon[0, 0]           # (H_out, W_out)

            if output_vol is None:
                H_out, W_out = recon_slice.shape
                output_vol = torch.zeros(
                    (D, H_out, W_out),
                    dtype=recon_slice.dtype,
                    device=device,
                )

            output_vol[i] = recon_slice

    # Return (1,1,D,H_out,W_out)
    return output_vol.unsqueeze(0).unsqueeze(0)
    return output_vol.unsqueeze(0).unsqueeze(0)

In [8]:
def center_crop_to_smallest(*vols):
    """
    Center-crop all volumes (B,C,D,H,W) to the smallest common (D,H,W).
    Assumes all volumes have same B and C.
    """
    # Get min spatial sizes
    Ds, Hs, Ws = zip(*(v.shape[2:] for v in vols))
    D_min, H_min, W_min = min(Ds), min(Hs), min(Ws)

    cropped = []
    for v in vols:
        _, _, D, H, W = v.shape
        d_start = (D - D_min) // 2
        h_start = (H - H_min) // 2
        w_start = (W - W_min) // 2
        cropped.append(
            v[:, :, d_start:d_start + D_min,
                 h_start:h_start + H_min,
                 w_start:w_start + W_min]
        )
    return cropped

Calculate average loss over the entire test set

In [ ]:
average_2d_loss = 0.0
average_3d_loss = 0.0
num_pairs = len(test_pairs)

for i, (hunt3_path, hunt4_path) in enumerate(tqdm(test_pairs, total=num_pairs)):
    # Load full volumes as NumPy arrays: either (H, W, D) or (D, H, W)
    x_vol_np = data_loader.load_from_path(hunt3_path, crop_size=(192, 224))
    y_vol_np = data_loader.load_from_path(hunt4_path, crop_size=(192, 224))

    # Ensure shape (D, H, W). Assumes crop_size=(H,W)=(192,224)
    if x_vol_np.shape[0] != 192 and x_vol_np.shape[2] == 192:
        # (H, W, D) -> (D, H, W)
        x_vol_np = np.transpose(x_vol_np, (2, 0, 1))
        y_vol_np = np.transpose(y_vol_np, (2, 0, 1))

    # (D,H,W) -> (1,1,D,H,W)
    x_vol = torch.from_numpy(x_vol_np).float().unsqueeze(0).unsqueeze(0).to(device)
    y_vol = torch.from_numpy(y_vol_np).float().unsqueeze(0).unsqueeze(0).to(device)

    # 2D model: reconstruct volume slice-by-slice
    recon_2d_vol = get_volume_from_2d_pred(best_2d, x_vol)

    # 3D model: direct full-volume prediction
    recon_3d_vol, _ = best_3d(x_vol)

    # --- Align shapes (B,C,D,H,W) via center crop to smallest size ---
    recon_2d_vol, recon_3d_vol, y_vol_aligned = center_crop_to_smallest(
        recon_2d_vol, recon_3d_vol, y_vol
    )

    # Compute losses
    average_2d_loss += recon_loss(recon_2d_vol, y_vol_aligned).item()
    average_3d_loss += recon_loss(recon_3d_vol, y_vol_aligned).item()

average_2d_loss /= num_pairs
average_3d_loss /= num_pairs

print(f"  Average 2D U-Net Test Error:      {average_2d_loss:.4f}")
print(f"  Average 3D U-Net Test Error:      {average_3d_loss:.4f}")

100%|██████████| 71/71 [00:56<00:00,  1.27it/s]


## Compare loss over entire volume for both models

As the 2D model only generates slices, it has to be run individually over all slices in a volume